# Resumen Intervencional: Beta=0 vs Beta óptima

Igual que en `Resumen_Observacional.ipynb`, pero sobre los 8 experimentos **Intervencional** (4 ruidos
aditivos + 4 multiplicativos): compara el modelo base (`Beta=0`) frente al modelo con la `Beta` óptima
elegida por el mismo criterio multi-métrica, y comprueba con Wilcoxon pareado si las mejoras son
significativas.

Diferencias frente al caso observacional:
- **No se incluye `HSIC(Z,Y)`**: bajo intervención, Y se fija externamente (`do(Y)`) y ya no se genera
  por su ecuación estructural, así que probar la independencia del residuo de Z frente a Y no tiene
  sentido (de hecho la columna `HSIC(Z,Y)` de estos CSVs viene vacía). Las métricas usadas son
  `MAE Z`, `HSIC(Z,X)` y `RF Acc`.
- **Nueva dimensión `Y_do`**: cada fila corresponde a `do(Y)=0.0` o `do(Y)=1.0`. Se calcula todo por
  separado para cada intervención, y además una versión "media" que agrupa ambas.
- Los 4 CSVs `multiplicativo_*` solo tienen 10 seeds (frente a 20 en `aditivo_*`), así que sus tests
  de Wilcoxon tienen menos potencia.

Todo el análisis se repite para `N=50` y `N=100`.


In [61]:
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"

ALPHA = 0.05
METRICAS = ["MAE Z", "HSIC(Z,X)", "RF Acc"]  # menor es mejor en las 3; sin HSIC(Z,Y) (no aplica bajo do(Y))


## Definición de los 8 experimentos intervencionales

In [62]:
EXPERIMENTOS = {
    "Aditivo Gaussiano": NOTEBOOKS_DIR / "Experimento1/Intervencional-20/tablas/intervencional_aditivo_gausian.csv",
    "Aditivo Exponencial": NOTEBOOKS_DIR / "Experimento1/Intervencional-20/tablas/intervencional_aditivo_exponencial.csv",
    "Aditivo Gamma": NOTEBOOKS_DIR / "Experimento1/Intervencional-20/tablas/intervencional_aditivo_gamma.csv",
    "Aditivo Uniforme": NOTEBOOKS_DIR / "Experimento1/Intervencional-20/tablas/intervencional_aditivo_uniforme.csv",
    "Multiplicativo Gaussiano": NOTEBOOKS_DIR / "Experimento2/Intervencional/tablas/intervencional_multiplicativo_gausian.csv",
    "Multiplicativo Exponencial": NOTEBOOKS_DIR / "Experimento2/Intervencional/tablas/intervencional_multiplicativo_exponencial.csv",
    "Multiplicativo Gamma": NOTEBOOKS_DIR / "Experimento2/Intervencional/tablas/intervencional_multiplicativo_gamma.csv",
    "Multiplicativo Uniforme": NOTEBOOKS_DIR / "Experimento2/Intervencional/tablas/intervencional_multiplicativo_uniforme.csv",
}

for nombre, path in EXPERIMENTOS.items():
    assert path.exists(), f"No existe: {path}"


## Función de selección de la Beta óptima

In [63]:
def beta_optima(df: pd.DataFrame, n_filter: int, y_do_filter, metrics=METRICAS):
    """Selecciona la Beta óptima (Beta != 0) de un experimento según un criterio multi-métrica.

    Beta=0 se usa solo como referencia (baseline) y nunca puede ser el resultado.
    `y_do_filter`: 0.0 o 1.0 para quedarse con esa intervención, o None para agrupar ambas (media).

    1) Filtra N == n_filter (y Y_do == y_do_filter si no es None), promedia cada métrica por Beta
       (sobre las seeds -y, si aplica, ambos Y_do- disponibles) y descarta Beta=0 de las candidatas.
    2) Para cada métrica, halla la Beta > 0 que minimiza su media -> betas candidatas.
    3) Para cada beta candidata, cuenta cuántas métricas mejoran/empeoran frente a Beta=0 y calcula
       la mejora relativa (%) de cada métrica frente al baseline.
    4) Se ordenan las candidatas por: (a) menos métricas empeoradas, (b) más métricas mejoradas,
       (c) mayor mejora relativa total, (d) último desempate (solo si empatan exactamente):
       mayor mejora relativa en una única métrica. Se elige la primera.

    Devuelve (baseline, beta_opt, valores_opt, candidatas, criterio).
    """
    df_n = df[df["N"] == n_filter]
    if y_do_filter is not None:
        df_n = df_n[df_n["Y_do"] == y_do_filter]

    media_por_beta = df_n.groupby("Beta")[list(metrics)].mean()

    baseline = media_por_beta.loc[0.0]
    media_no_cero = media_por_beta.drop(index=0.0)

    betas_candidatas = sorted(set(media_no_cero[m].idxmin() for m in metrics))

    filas_candidatas = []
    for beta in betas_candidatas:
        valores = media_no_cero.loc[beta]
        mejora_relativa = (baseline - valores) / baseline
        filas_candidatas.append({
            "Beta": beta,
            "n_mejoradas": int((valores < baseline).sum()),
            "n_empeoradas": int((valores > baseline).sum()),
            "mejora_relativa_total": float(mejora_relativa.sum()),
            "mejora_relativa_max": float(mejora_relativa.max()),
        })

    candidatas = pd.DataFrame(filas_candidatas).sort_values(
        by=["n_empeoradas", "n_mejoradas", "mejora_relativa_total", "mejora_relativa_max"],
        ascending=[True, False, False, False],
    ).reset_index(drop=True)

    ganadora = candidatas.iloc[0]
    beta_opt = float(ganadora["Beta"])
    valores_opt = media_por_beta.loc[beta_opt]

    if ganadora["n_empeoradas"] == 0:
        criterio = "domina al baseline (mejora >=1 metrica sin empeorar ninguna)"
    else:
        criterio = "mejor compromiso (ninguna beta evita empeorar alguna metrica)"

    return baseline, beta_opt, valores_opt, candidatas, criterio


## Función de test de Wilcoxon pareado

Se empareja por `(Seed, Y_do)` en vez de solo `Seed`: así funciona igual si se filtra a un único
`Y_do` (queda un valor de `Y_do` por seed, como antes) o si se agrupan ambos (quedan 2 filas por seed,
una por intervención, cada una emparejada con su contraparte en la misma condición).


In [64]:
def wilcoxon_experimento(df: pd.DataFrame, beta_opt: float, n_filter: int, y_do_filter, metrics=METRICAS):
    """Wilcoxon signed-rank pareado entre Beta=0 y Beta=beta_opt, por métrica.

    H0: no hay diferencia. H1 (alternative='less'): el valor en beta_opt es menor (mejor) que en Beta=0.
    Devuelve (p_valores: dict metrica->p, n_pares: int).
    """
    df_n = df[df["N"] == n_filter]
    if y_do_filter is not None:
        df_n = df_n[df_n["Y_do"] == y_do_filter]

    base = df_n[df_n["Beta"] == 0.0].set_index(["Seed", "Y_do"])[list(metrics)]
    opt = df_n[df_n["Beta"] == beta_opt].set_index(["Seed", "Y_do"])[list(metrics)]
    pares_comunes = sorted(set(base.index) & set(opt.index))
    base = base.loc[pares_comunes]
    opt = opt.loc[pares_comunes]

    p_valores = {}
    for m in metrics:
        try:
            _, p = wilcoxon(opt[m].values, base[m].values, alternative="less")
        except ValueError:
            p = float("nan")
        p_valores[m] = p
    return p_valores, len(pares_comunes)


## Cálculo (tabla resumen + p-valores) para un (N, do(Y)) dado

In [65]:
def analizar(n_filter: int, y_do_filter):
    """Ejecuta beta_optima + wilcoxon_experimento para los 8 experimentos, a N y Y_do fijos.

    y_do_filter: 0.0, 1.0, o None (media entre ambas intervenciones).
    Devuelve (resumen: DataFrame, p_values: DataFrame, diagnostico: dict[str, DataFrame]).
    """
    filas_resumen = []
    filas_p = []
    diagnostico = {}

    for nombre, path in EXPERIMENTOS.items():
        df = pd.read_csv(path)
        baseline, beta_opt, valores_opt, candidatas, criterio = beta_optima(df, n_filter, y_do_filter)
        diagnostico[nombre] = candidatas

        filas_resumen.append({
            "Experimento": nombre,
            "beta_optima": beta_opt,
            "MAE(Z) beta=0": baseline["MAE Z"],
            "MAE(Z) beta_optima": valores_opt["MAE Z"],
            "HSIC(Z,X) beta=0": baseline["HSIC(Z,X)"],
            "HSIC(Z,X) beta_optima": valores_opt["HSIC(Z,X)"],
            "RF Acc beta=0": baseline["RF Acc"],
            "RF Acc beta_optima": valores_opt["RF Acc"],
            "Criterio": criterio,
        })

        p_valores, n_pares = wilcoxon_experimento(df, beta_opt, n_filter, y_do_filter)
        filas_p.append({"Experimento": nombre, **p_valores, "n_pares": n_pares})

    resumen = pd.DataFrame(filas_resumen)
    p_values = pd.DataFrame(filas_p).set_index("Experimento")
    return resumen, p_values, diagnostico


resultados = {}
for n_filter in (50, 100):
    for y_do_filter, etiqueta in ((0.0, "ydo0"), (1.0, "ydo1"), (None, "media")):
        resultados[(n_filter, etiqueta)] = analizar(n_filter, y_do_filter)


## Funciones de formato y resaltado en negrita

In [66]:
COLUMNA_A_METRICA = {
    "MAE(Z) beta_optima": "MAE Z",
    "HSIC(Z,X) beta_optima": "HSIC(Z,X)",
    "RF Acc beta_optima": "RF Acc",
}


def formatear(resumen: pd.DataFrame):
    cols_num = [c for c in resumen.columns if c not in ("Experimento", "beta_optima", "Criterio")]
    fmt = resumen.copy()
    fmt[cols_num] = fmt[cols_num].round(5)
    fmt["beta_optima"] = fmt["beta_optima"].round(2)
    return fmt


def tabla_con_negrita(resumen: pd.DataFrame, p_values: pd.DataFrame):
    """Tabla resumen con las celdas 'beta_optima' en negrita cuando su p-valor (Wilcoxon) < ALPHA.

    Solo aplica en la visualización del notebook (un CSV plano no admite negrita).
    """
    fmt = formatear(resumen)

    def resaltar(row):
        p_exp = p_values.loc[row["Experimento"]]
        estilos = []
        for col in row.index:
            metrica = COLUMNA_A_METRICA.get(col)
            if metrica is not None and pd.notna(p_exp[metrica]) and p_exp[metrica] < ALPHA:
                estilos.append("font-weight: bold")
            else:
                estilos.append("")
        return estilos

    return fmt.style.apply(resaltar, axis=1)


## N = 50

### do(Y) = 0.0

In [67]:
resumen, p_values = resultados[(50, "ydo0")][0], resultados[(50, "ydo0")][1]
tabla_con_negrita(resumen, p_values)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio
0,Aditivo Gaussiano,0.100000,1.021620,1.015550,0.088420,0.084770,0.586250,0.553750,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
1,Aditivo Exponencial,0.100000,0.984260,0.990990,0.122020,0.120060,0.666250,0.666250,mejor compromiso (ninguna beta evita empeorar alguna metrica)
2,Aditivo Gamma,0.300000,1.009840,0.978040,0.222560,0.199740,0.750000,0.728750,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
3,Aditivo Uniforme,0.100000,0.966670,0.956870,0.073490,0.066490,0.627500,0.610000,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
4,Multiplicativo Gaussiano,0.900000,1.063700,1.001130,0.207960,0.143020,0.672500,0.650000,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
5,Multiplicativo Exponencial,0.900000,0.752580,0.731180,0.212770,0.093160,0.700000,0.672500,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
6,Multiplicativo Gamma,0.600000,0.826100,0.691170,0.244680,0.184780,0.702500,0.645000,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
7,Multiplicativo Uniforme,0.500000,0.923970,0.913340,0.098640,0.074130,0.570000,0.590000,mejor compromiso (ninguna beta evita empeorar alguna metrica)


In [68]:
p_values.round(4)


,MAE Z,"HSIC(Z,X)",RF Acc,n_pares
Experimento,,,,
Aditivo Gaussiano,0.4062,0.0527,0.0006,20
Aditivo Exponencial,0.1227,0.1559,0.2806,20
Aditivo Gamma,0.0825,0.0947,0.0766,20
Aditivo Uniforme,0.1567,0.0181,0.1189,20
Multiplicativo Gaussiano,0.1377,0.0967,0.1328,10
Multiplicativo Exponencial,0.1377,0.0029,0.2969,10
Multiplicativo Gamma,0.0322,0.1377,0.0039,10
Multiplicativo Uniforme,0.0420,0.0244,0.9453,10


### do(Y) = 1.0

In [69]:
resumen, p_values = resultados[(50, "ydo1")][0], resultados[(50, "ydo1")][1]
tabla_con_negrita(resumen, p_values)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio
0,Aditivo Gaussiano,0.500000,0.921630,0.902310,0.062410,0.049740,0.582500,0.560000,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
1,Aditivo Exponencial,0.100000,0.814900,0.815790,0.102290,0.096770,0.665000,0.665000,mejor compromiso (ninguna beta evita empeorar alguna metrica)
2,Aditivo Gamma,0.100000,0.851990,0.845250,0.171350,0.160560,0.713750,0.723750,mejor compromiso (ninguna beta evita empeorar alguna metrica)
3,Aditivo Uniforme,0.400000,0.925750,0.929120,0.048810,0.045150,0.607500,0.615000,mejor compromiso (ninguna beta evita empeorar alguna metrica)
4,Multiplicativo Gaussiano,0.700000,0.712610,0.599940,0.352430,0.165110,0.660000,0.640000,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
5,Multiplicativo Exponencial,0.400000,0.538480,0.454620,0.354820,0.209800,0.760000,0.725000,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
6,Multiplicativo Gamma,0.700000,0.580750,0.480450,0.406940,0.262150,0.755000,0.715000,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
7,Multiplicativo Uniforme,0.100000,0.528090,0.523000,0.154660,0.140800,0.567500,0.565000,domina al baseline (mejora >=1 metrica sin empeorar ninguna)


In [70]:
p_values.round(4)


,MAE Z,"HSIC(Z,X)",RF Acc,n_pares
Experimento,,,,
Aditivo Gaussiano,0.3238,0.0107,0.0242,20
Aditivo Exponencial,0.7271,0.0148,0.5475,20
Aditivo Gamma,0.2105,0.0014,0.8455,20
Aditivo Uniforme,0.4347,0.0615,0.6913,20
Multiplicativo Gaussiano,0.0029,0.0020,0.2637,10
Multiplicativo Exponencial,0.0020,0.0322,0.1523,10
Multiplicativo Gamma,0.0322,0.0322,0.0664,10
Multiplicativo Uniforme,0.3477,0.1162,0.3750,10


### Media entre do(Y)=0.0 y do(Y)=1.0

In [71]:
resumen, p_values = resultados[(50, "media")][0], resultados[(50, "media")][1]
tabla_con_negrita(resumen, p_values)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio
0,Aditivo Gaussiano,0.400000,0.971630,0.961710,0.075420,0.067120,0.584380,0.569380,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
1,Aditivo Exponencial,0.100000,0.899580,0.903390,0.112160,0.108410,0.665620,0.665620,mejor compromiso (ninguna beta evita empeorar alguna metrica)
2,Aditivo Gamma,0.300000,0.930920,0.925250,0.196960,0.176050,0.731880,0.723120,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
3,Aditivo Uniforme,0.200000,0.946210,0.942870,0.061150,0.055910,0.617500,0.619380,mejor compromiso (ninguna beta evita empeorar alguna metrica)
4,Multiplicativo Gaussiano,0.900000,0.888160,0.832470,0.280190,0.190210,0.666250,0.657500,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
5,Multiplicativo Exponencial,0.900000,0.645530,0.632900,0.283790,0.133270,0.730000,0.693750,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
6,Multiplicativo Gamma,0.700000,0.703430,0.586340,0.325810,0.223690,0.728750,0.683750,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
7,Multiplicativo Uniforme,0.500000,0.726030,0.720040,0.126650,0.113210,0.568750,0.566250,domina al baseline (mejora >=1 metrica sin empeorar ninguna)


In [72]:
p_values.round(4)


,MAE Z,"HSIC(Z,X)",RF Acc,n_pares
Experimento,,,,
Aditivo Gaussiano,0.1505,0.0218,0.0386,40
Aditivo Exponencial,0.2025,0.0104,0.4459,40
Aditivo Gamma,0.3673,0.0108,0.2400,40
Aditivo Uniforme,0.2340,0.0265,0.5819,40
Multiplicativo Gaussiano,0.2045,0.0242,0.2763,20
Multiplicativo Exponencial,0.2262,0.0000,0.0528,20
Multiplicativo Gamma,0.0086,0.0164,0.0044,20
Multiplicativo Uniforme,0.0487,0.1012,0.5208,20


## N = 100

### do(Y) = 0.0

In [73]:
resumen, p_values = resultados[(100, "ydo0")][0], resultados[(100, "ydo0")][1]
tabla_con_negrita(resumen, p_values)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio
0,Aditivo Gaussiano,0.200000,0.930230,0.916410,0.064160,0.054500,0.563750,0.566250,mejor compromiso (ninguna beta evita empeorar alguna metrica)
1,Aditivo Exponencial,0.300000,0.915990,0.841420,0.109420,0.081990,0.656250,0.643750,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
2,Aditivo Gamma,0.100000,0.918570,0.894390,0.196480,0.178170,0.722500,0.722500,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
3,Aditivo Uniforme,0.100000,0.946440,0.934190,0.050640,0.045830,0.603750,0.597500,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
4,Multiplicativo Gaussiano,0.200000,0.945350,0.929550,0.104540,0.088950,0.710000,0.692500,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
5,Multiplicativo Exponencial,0.500000,0.752200,0.724070,0.198910,0.147780,0.690000,0.682500,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
6,Multiplicativo Gamma,0.600000,0.714910,0.637770,0.168520,0.096920,0.687500,0.662500,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
7,Multiplicativo Uniforme,0.400000,0.905090,0.905210,0.084290,0.084160,0.627500,0.617500,mejor compromiso (ninguna beta evita empeorar alguna metrica)


In [74]:
p_values.round(4)


,MAE Z,"HSIC(Z,X)",RF Acc,n_pares
Experimento,,,,
Aditivo Gaussiano,0.1942,0.1227,0.6732,20
Aditivo Exponencial,0.0010,0.0002,0.2791,20
Aditivo Gamma,0.0884,0.0220,0.7087,20
Aditivo Uniforme,0.2045,0.0825,0.2384,20
Multiplicativo Gaussiano,0.2461,0.0654,0.1172,10
Multiplicativo Exponencial,0.0098,0.0186,0.4375,10
Multiplicativo Gamma,0.0244,0.1611,0.4492,10
Multiplicativo Uniforme,0.5000,0.5000,0.5391,10


### do(Y) = 1.0

In [75]:
resumen, p_values = resultados[(100, "ydo1")][0], resultados[(100, "ydo1")][1]
tabla_con_negrita(resumen, p_values)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio
0,Aditivo Gaussiano,0.200000,0.908590,0.909440,0.039970,0.037210,0.567500,0.552500,mejor compromiso (ninguna beta evita empeorar alguna metrica)
1,Aditivo Exponencial,0.300000,0.833220,0.812980,0.091240,0.063780,0.646250,0.662500,mejor compromiso (ninguna beta evita empeorar alguna metrica)
2,Aditivo Gamma,0.300000,0.790090,0.821580,0.128340,0.099990,0.692500,0.708750,mejor compromiso (ninguna beta evita empeorar alguna metrica)
3,Aditivo Uniforme,0.100000,0.902680,0.896970,0.034410,0.033150,0.580000,0.566250,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
4,Multiplicativo Gaussiano,0.200000,0.550600,0.544330,0.142470,0.128900,0.642500,0.642500,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
5,Multiplicativo Exponencial,0.200000,0.477380,0.463000,0.312820,0.238760,0.725000,0.717500,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
6,Multiplicativo Gamma,0.700000,0.461700,0.376490,0.260980,0.113830,0.725000,0.690000,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
7,Multiplicativo Uniforme,0.800000,0.511250,0.535110,0.116390,0.159620,0.547500,0.545000,mejor compromiso (ninguna beta evita empeorar alguna metrica)


In [76]:
p_values.round(4)


,MAE Z,"HSIC(Z,X)",RF Acc,n_pares
Experimento,,,,
Aditivo Gaussiano,0.7392,0.4062,0.0918,20
Aditivo Exponencial,0.1305,0.0000,0.9178,20
Aditivo Gamma,0.9552,0.0120,0.8324,20
Aditivo Uniforme,0.2979,0.2729,0.0844,20
Multiplicativo Gaussiano,0.6523,0.5771,0.4727,10
Multiplicativo Exponencial,0.0967,0.0068,0.4219,10
Multiplicativo Gamma,0.0801,0.0654,0.1211,10
Multiplicativo Uniforme,0.9678,0.9346,0.4395,10


### Media entre do(Y)=0.0 y do(Y)=1.0

In [77]:
resumen, p_values = resultados[(100, "media")][0], resultados[(100, "media")][1]
tabla_con_negrita(resumen, p_values)


,Experimento,beta_optima,MAE(Z) beta=0,MAE(Z) beta_optima,"HSIC(Z,X) beta=0","HSIC(Z,X) beta_optima",RF Acc beta=0,RF Acc beta_optima,Criterio
0,Aditivo Gaussiano,0.200000,0.919410,0.912930,0.052070,0.045860,0.565630,0.559370,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
1,Aditivo Exponencial,0.300000,0.874600,0.827200,0.100330,0.072880,0.651250,0.653120,mejor compromiso (ninguna beta evita empeorar alguna metrica)
2,Aditivo Gamma,0.100000,0.854330,0.852040,0.162410,0.144810,0.707500,0.710620,mejor compromiso (ninguna beta evita empeorar alguna metrica)
3,Aditivo Uniforme,0.100000,0.924560,0.915580,0.042520,0.039490,0.591880,0.581870,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
4,Multiplicativo Gaussiano,0.200000,0.747980,0.736940,0.123500,0.108920,0.676250,0.667500,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
5,Multiplicativo Exponencial,0.400000,0.614790,0.591380,0.255860,0.196420,0.707500,0.697500,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
6,Multiplicativo Gamma,0.700000,0.588310,0.505360,0.214750,0.107550,0.706250,0.675000,domina al baseline (mejora >=1 metrica sin empeorar ninguna)
7,Multiplicativo Uniforme,0.200000,0.708170,0.709470,0.100340,0.104950,0.587500,0.580000,mejor compromiso (ninguna beta evita empeorar alguna metrica)


In [78]:
p_values.round(4)


,MAE Z,"HSIC(Z,X)",RF Acc,n_pares
Experimento,,,,
Aditivo Gaussiano,0.4393,0.1569,0.2921,40
Aditivo Exponencial,0.0011,0.0000,0.6536,40
Aditivo Gamma,0.4973,0.0025,0.7841,40
Aditivo Uniforme,0.1474,0.0750,0.0483,40
Multiplicativo Gaussiano,0.3371,0.1942,0.2200,20
Multiplicativo Exponencial,0.0018,0.0002,0.2646,20
Multiplicativo Gamma,0.0053,0.0242,0.0769,20
Multiplicativo Uniforme,0.3238,0.4062,0.2490,20


## Comparación entre las 6 combinaciones (N x do(Y))

Beta óptima elegida y número de métricas significativas (de 3: `MAE Z`, `HSIC(Z,X)`, `RF Acc`) en
cada combinación, para ver de un vistazo qué cambia con N y con la intervención.


In [79]:
combos = [(50, "ydo0"), (50, "ydo1"), (50, "media"), (100, "ydo0"), (100, "ydo1"), (100, "media")]

comparacion = pd.DataFrame({"Experimento": list(EXPERIMENTOS.keys())})
for n_filter, etiqueta in combos:
    resumen_c, p_values_c, _ = resultados[(n_filter, etiqueta)]
    col_beta = f"beta_optima N={n_filter} {etiqueta}"
    col_sig = f"metricas_sig N={n_filter} {etiqueta}"
    comparacion = comparacion.merge(
        resumen_c[["Experimento", "beta_optima"]].rename(columns={"beta_optima": col_beta}),
        on="Experimento",
    )
    comparacion[col_sig] = comparacion["Experimento"].map((p_values_c[METRICAS] < ALPHA).sum(axis=1))

comparacion


,Experimento,beta_optima N=50 ydo0,metricas_sig N=50 ydo0,beta_optima N=50 ydo1,metricas_sig N=50 ydo1,beta_optima N=50 media,metricas_sig N=50 media,beta_optima N=100 ydo0,metricas_sig N=100 ydo0,beta_optima N=100 ydo1,metricas_sig N=100 ydo1,beta_optima N=100 media,metricas_sig N=100 media
0,Aditivo Gaussiano,0.1,1,0.5,2,0.4,2,0.2,0,0.2,0,0.2,0
1,Aditivo Exponencial,0.1,0,0.1,1,0.1,1,0.3,2,0.3,1,0.3,2
2,Aditivo Gamma,0.3,0,0.1,1,0.3,1,0.1,1,0.3,1,0.1,1
3,Aditivo Uniforme,0.1,1,0.4,0,0.2,1,0.1,0,0.1,0,0.1,1
4,Multiplicativo Gaussiano,0.9,0,0.7,2,0.9,1,0.2,0,0.2,0,0.2,0
5,Multiplicativo Exponencial,0.9,1,0.4,2,0.9,1,0.5,2,0.2,1,0.4,2
6,Multiplicativo Gamma,0.6,2,0.7,2,0.7,3,0.6,1,0.7,0,0.7,2
7,Multiplicativo Uniforme,0.5,2,0.1,0,0.5,1,0.4,0,0.8,0,0.2,0


## Guardar CSVs resumen

In [80]:
for (n_filter, etiqueta), (resumen_c, _, _) in resultados.items():
    out_path = NOTEBOOKS_DIR / "tablas" / f"resumen_intervencional_beta_n{n_filter}_{etiqueta}.csv"
    resumen_c.to_csv(out_path, index=False)
    print(f"Guardado en: {out_path}")


Guardado en: /Users/clau/Documents/TFM_NUEVO/CODIGO_COMPLETO/kacgm-hsic/kacgm_hsic/notebooks/tablas/resumen_intervencional_beta_n50_ydo0.csv
Guardado en: /Users/clau/Documents/TFM_NUEVO/CODIGO_COMPLETO/kacgm-hsic/kacgm_hsic/notebooks/tablas/resumen_intervencional_beta_n50_ydo1.csv
Guardado en: /Users/clau/Documents/TFM_NUEVO/CODIGO_COMPLETO/kacgm-hsic/kacgm_hsic/notebooks/tablas/resumen_intervencional_beta_n50_media.csv
Guardado en: /Users/clau/Documents/TFM_NUEVO/CODIGO_COMPLETO/kacgm-hsic/kacgm_hsic/notebooks/tablas/resumen_intervencional_beta_n100_ydo0.csv
Guardado en: /Users/clau/Documents/TFM_NUEVO/CODIGO_COMPLETO/kacgm-hsic/kacgm_hsic/notebooks/tablas/resumen_intervencional_beta_n100_ydo1.csv
Guardado en: /Users/clau/Documents/TFM_NUEVO/CODIGO_COMPLETO/kacgm-hsic/kacgm_hsic/notebooks/tablas/resumen_intervencional_beta_n100_media.csv
